In [1]:
%matplotlib widget
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import sys

sys.path.append('/home/wolfgang/repos/ICU-Sleep/code1')
from airgo_features import airgo_actigraphy_features

sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file, read_in_routine

from collections import Counter
from scipy import io as sio
import scipy

from tqdm import tqdm

from sleep_staging_functions import *


%load_ext autoreload
%autoreload 2

In [2]:
def check_load_mgh_dataset(signalfilepath, labelfilepath, channels=None, report_and_actual_time_tol=300, reverse_sign=False):

    gender = None
    handedness = None
    
    try: # usually Grass, saved as pre 7.3 format:
        ff = sio.loadmat(signalfilepath)
        data_path = os.path.basename(signalfilepath)
        if 's' not in ff:
            raise Exception('No signal found in %s.'%data_path)
        signal = ff['s']
        if reverse_sign:
            signal = -signal
        channel_names = [ff['hdr'][0,i]['signal_labels'][0].upper().replace('EKG','ECG') for i in range(ff['hdr'].shape[1])]
        gender = None
        handedness = None
        if 'grass' in signalfilepath:
            Fs = 200.
        else:
            raise Exception('Safety check to make sure this is Grass data with Fs=200')
        # Label part for grass:
        if 'grass' in signalfilepath:
            # load labels
            with h5py.File(labelfilepath) as ffl:
                sleep_stage = ffl['stage'][()].flatten()
                start_time, end_time = get_grass_start_end_time(ffl['features']['StartTime'][()].flatten(), ffl['features']['EndTime'][()].flatten())


    except: # saved as .mat 7.3. new grass or natus files:
        with h5py.File(signalfilepath,'r') as ff:

            hdr = ff['hdr']
            signal_labels = hdr['signal_labels'][:]
            channel_names = [ ''.join(list(map(chr, ff[signal_labels[i,0]][:]))) for i in range(signal_labels.shape[0]) ]
            channel_names = [channel.upper() for channel in channel_names]
            signal = ff['s'][()]
            signal = np.transpose(signal);

            if 'recording' in ff.keys(): # only available for natus:
                year = int(np.squeeze(ff['recording']['year']))
                month = int(np.squeeze(ff['recording']['month']))
                day = int(np.squeeze(ff['recording']['day']))
                hour = int(np.squeeze(ff['recording']['hour']))
                if (hour >= 7) and (hour <=10):         # 'typo' by sleep techs
                    hour = hour + 12
                minute = int(np.squeeze(ff['recording']['minute']))
                second = int(np.squeeze(ff['recording']['second']))
                millisecond = int(np.squeeze(ff['recording']['millisecond']))
                Fs = int(np.squeeze(ff['recording']['samplingrate']))

                start_time = datetime.datetime(1990,1,1,hour=hour, minute=minute,
                        second=second, microsecond=int(millisecond*1000))
                end_time = start_time+datetime.timedelta(seconds=max(signal.shape)/Fs)

            else: # grass:
                if 'grass' in signalfilepath:
                    Fs = 200.
                else:
                    raise Exception('Safety check to make sure this is Grass data with Fs=200')

            # Label part for grass:
            ff.close()

        # load labels
        with h5py.File(labelfilepath) as ffl:
            sleep_stage = ffl['stage'][()].flatten()
            if 'grass' in signalfilepath:
                start_time, end_time = get_grass_start_end_time(ffl['features']['StartTime'][()].flatten(), ffl['features']['EndTime'][()].flatten())
            ffl.close()

    # end of loading part
    ##################################
    
    # check signal length = sleep stage length
    if sleep_stage.shape[0]!=signal.shape[1]:
        raise Exception('Inconsistent sleep stage length (%d) and signal length (%d) in %s'%(sleep_stage.shape[0],signal.shape[1],data_path))

    # only take signal channels to study
    if channels is None:
        signal_channel_ids = list(range(len(channel_names)))
        
    elif 'SumEffort' in channels:
        signal_channel_ids = []
        for ichannel in ['ABD','CHEST']:
            #channel_name_pattern = re.compile(channels[i][:2].upper()+'-*'+channels[i][-2:].upper())
            found = False
            for j in range(len(channel_names)):
                if channel_names[j]==ichannel.upper():
                    signal_channel_ids.append(j)
                    found = True
                    break
            if not found:
                raise Exception('Channel %s is not found.'%ichannel)
        signal = signal[signal_channel_ids,:]#.T
        # do effort belt average here:
        signal = np.sum(signal,0,keepdims=1)/2
        
    else:
        signal_channel_ids = []
        for i in range(len(channels)):
            #channel_name_pattern = re.compile(channels[i][:2].upper()+'-*'+channels[i][-2:].upper())
            found = False
            for j in range(len(channel_names)):
                if channel_names[j]==channels[i].upper():
                    signal_channel_ids.append(j)
                    found = True
                    break
            if not found:
                raise Exception('Channel %s is not found.'%channels[i])
        signal = signal[signal_channel_ids,:]

    # check whether the signal contains NaN
    if np.any(np.isnan(signal)):
        raise Exception('Found Nan in signal in %s'%data_path)

    # check whether sleep_stage contains all 5 stages
    stages = np.unique(sleep_stage[np.logical_not(np.isnan(sleep_stage))]).astype(int).tolist()
    if len(stages)<=1:
        raise Exception('#sleep stage <= 1: %s in %s'%(stages,data_path))

    params = {'Fs':Fs*1.0, 'signal_channel_names': channel_names, 'signal_channel_ids':signal_channel_ids}
    if gender is not None:
        params['gender'] = gender
    if handedness is not None:
        params['handedness'] = handedness
    return signal, sleep_stage, params



In [5]:
from sleep_staging_functions import ecg_peak_detection_sqi_plot, apply_ecg_sleep_staging_models, ecg_peak_finding_physionet

### Sleeplab data:

In [6]:
# input_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/biosignals_10Hz_data'
input_dir = '/media/mad3/Datasets_ConvertedData/sleeplab/natus_data/Arroyo~ Brenda_6dad51c7-ec95-48fb-b9d7-fa288efbbe6b'
output_dir = '/media/ssd2/icu_sleepstaging'

verbose = False
files = os.listdir(input_dir)
files.sort()

signal_file_name  = [x for x in files if 'Signal_' in x][0]
label_file_name = [x for x in files if 'Labels_' in x][0]
data_path = os.path.join(input_dir, signal_file_name)
label_path = os.path.join(input_dir, label_file_name)

if 0:
    signal1, sleep_stage, params = check_load_mgh_dataset(data_path, label_path, channels=['ECG-LA'])
    
    fs = int(params['Fs'])
    original_fs = int(params['Fs'])
    print(fs)
    
    signal = mne.filter.notch_filter(signal1, fs, np.arange(60,241,60), notch_widths=np.arange(60,241,60)/500)
    
    new_fs = 200
    
    signal = signal.flatten()
    signal.shape
    
    signal = scipy.signal.resample_poly(signal, new_fs, fs)
    
    fs = new_fs
    fs
    
    signal2 = signal[2000:2000+new_fs*60*6]
    signal2 = pd.Series(signal2).rolling(2, center=True, min_periods=1).mean().values

    nn, sqi = ecg_peak_finding_physionet(signal)
    
    # add R peaks and SQI to 240Hz ECG data:
    signal_ecg_lead = pd.DataFrame()
    signal_ecg_lead['signal'] = signal

    signal_ecg_lead['r_peak'] = np.int16(0)
    signal_ecg_lead.loc[signal_ecg_lead.iloc[np.round(nn[:,0]*fs).astype(int)-1].index, 'r_peak'] = 1
    signal_ecg_lead.loc[signal_ecg_lead.iloc[np.round(nn[:,0]*fs).astype(int)-1].index, 'nn_interval'] = nn[:,1]
    signal_ecg_lead['sqi'] = np.nan
    signal_ecg_lead.loc[signal_ecg_lead.iloc[np.round(sqi[:, 0] * fs).astype(int) - 1].index, 'sqi'] = sqi[:, 1]
    signal_ecg_lead['sqi'] = signal_ecg_lead['sqi'].astype(np.float32)
    signal_ecg_lead['sqi'].fillna(method='ffill', limit = 30*fs, inplace=True)
    signal_ecg_lead['sqi'].fillna(method='bfill', limit = 30*fs, inplace=True)
    signal_ecg_lead['sqi'].fillna(method='bfill', limit = 30*fs, inplace=True)
    signal_ecg_lead['sqi'].fillna(0, inplace=True)
    
    data = signal_ecg_lead
    
    data, seg_start_pos, yp = apply_ecg_sleep_staging_models(data, fs=fs, model_ids=['ecg_nn'], input_signals=['r_peak'], verbose=True)

In [7]:
if 0:
    
    study_id ='000'

    start_idx = 0
    duration = 10000000

    ecg_peak_detection_sqi_plot()

In [8]:
# add expert label to data:

if 0:
    
    
    if 0:
        sleep_stage_orig = sleep_stage.copy()
    else: 
        sleep_stage = sleep_stage_orig.copy()

    sleep_stage = scipy.signal.resample_poly(sleep_stage, new_fs, original_fs)
    sleep_stage[pd.notna(sleep_stage)] = np.round(sleep_stage[pd.notna(sleep_stage)]).astype(int)

    data['stage_expert'] = sleep_stage

    
    fig, ax = plt.subplots(3,1,figsize=(10,6), sharex=True)
    ax[0].plot(data.stage_pred_ecg_nn, c='blue')
    ax[0].plot(data.stage_expert+0.1, c='orange')
    ax[1].plot(data.nn_interval.dropna())
    ax[2].plot(data.sqi)

### ICU data

In [57]:
### icu sleep data:
from HRV_and_CPC_analysis_functions import load_ecg_data
import re

icu_nn_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_resampled_ECG_and_rpeaks'
files = os.listdir(icu_nn_dir)

# save results of ECG sleep staging to 10hz research data:
directory_10hz = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/icu_files_step3_nohrv'
files_10hz = os.listdir(directory_10hz)
print(len(files_10hz))

savedir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/icu_files_step_5_ecg_sleepstaging_nohrv'
if not os.path.exists(savedir):
    os.makedirs(savedir)


new_fs = ecg_sleepstage_model_fs = 200

132


In [85]:
for file in files[:-1]:
    
    study_id = re.search('\d\d\d', file)[0]

    file_path = os.path.join(icu_nn_dir, file)
    data, hdr, fs = read_in_routine(file_path)

    data_result = ecg_sleep_staging_routine_icusleep(data, fs)

    file_10hz = 'icusleep_' + study_id + '.h5'
    if not file_10hz in files_10hz:
        print(f'10Hz data not available for {study_id}. Continue.')
    #     continue
    filepath_10hz = os.path.join(directory_10hz, file_10hz)
    data_10hz, hdr_10hz, fs_10hz = read_in_routine(filepath_10hz)

    stage_pred_ecg_nn_10hz = data_result.stage_pred_ecg_nn.resample('100ms').max()
    nn_interval = data_result.nn_interval.resample('100ms').mean()
    data_10hz = data_10hz.join(stage_pred_ecg_nn_10hz, how='left').join(nn_interval, how='left')

    write_to_hdf5_file(data_10hz, os.path.join(savedir, file_10hz), hdr_10hz)

In [38]:
data_10hz = data_10hz.join(stage_pred_ecg_nn_10hz, how='left').join(nn_interval, how='left')

In [53]:
# sleep stage comparison respiration, ecg, ai, also plot NN and RR

if 1:
    fig, ax = plt.subplots(4,1,figsize=(10,6), sharex=True)
    ax[0].plot(data_10hz.stage_pred_ecg_nn, c='red')
    ax[0].plot(data_10hz.stage_pred_amendsumeffort, c='blue')
    ax[0].plot(data_10hz.stage_pred_activity10sec, c='gray')
    
    ax[1].plot(data_10hz.nn_interval.dropna(), c='red')
    ax[2].plot(data_10hz.rr_10s_smooth, c='blue')
    ax[3].plot(data_10hz.acc_ai_10sec, c='gray')

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [46]:
[x for x in data_10hz.columns if 'stage' in x]

['stage_pred_activity10sec',
 'stage_pred_amendsumeffort',
 'stage_pred_comb_breath_activity_1',
 'stage_pred_ecg_nn']

In [32]:
# ECG, SQI and Sleep Stages Result:

if 0:
    fig, ax = plt.subplots(4,1,figsize=(10,6), sharex=True)
    ax[0].plot(data_result.stage_pred_ecg_nn, c='red')
    ax[1].plot(data_result.nn_interval.dropna())
    ax[2].plot(data_result.signal)
    ax[3].plot(data_result.sqi)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [9]:

# def segment_signal_only(signal, window_time, step_time, fs, 
#                         notch_freq=None, bandpass_freq=None, start_end_remove_window_num=1, 
#                         amplitude_thres=None, amplitude_thres_low=None, n_jobs=-1, 
#                         to_remove_mean=False, z_normalize_signal=True,
#                         std_thres1=0.00001, std_thres2=0.00001):

#     # segmentation:

#     window_size = int(round(window_time*fs))
#     step_size = int(round(step_time*fs))

#     # prepare segmentation:
#     start_ids = np.arange(0, signal.shape[1]-window_size+1, step_size)
#     if start_end_remove_window_num>0:
#         start_ids = start_ids[start_end_remove_window_num:-start_end_remove_window_num]
#     seg_masks = [seg_mask_explanation[0]]*len(start_ids)

#     ### check for high amplitude (airgo not worn) before detrending signal:
#     if amplitude_thres is not None:
#     ## find large amplitude in signal
#         signal_segs_temp = signal[:,[np.arange(x,x+window_size) for x in start_ids]].transpose(1,0,2)  # (#window, #ch, window_size+2padding)

#         amplitude_large2d = np.any(np.abs(signal_segs_temp)>amplitude_thres, axis=2)
#         amplitude_large1d = np.where(np.any(amplitude_large2d, axis=1))[0]
#         for i in amplitude_large1d:
#             seg_masks[i] = '%s'%(seg_mask_explanation[4])

#     if amplitude_thres_low is not None:
#         ## find low amplitude in signal
#         signal_segs_temp = signal[:,[np.arange(x,x+window_size) for x in start_ids]].transpose(1,0,2)  # (#window, #ch, window_size+2padding)
#         amplitude_low2d = np.any(np.abs(signal_segs_temp)<amplitude_thres_low, axis=2)
#         amplitude_low1d = np.where(np.any(amplitude_low2d, axis=1))[0]
#         for i in amplitude_low1d:
#             seg_masks[i] = '%s'%(seg_mask_explanation[4])

#     if z_normalize_signal:
#         # normalize signal:
#         signal = clip_z_normalize(pd.Series(signal[0,:]))
#         signal = signal[np.newaxis, :]

#     # create signal segments:
#     signal_segs = signal[:,[np.arange(x,x+window_size) for x in start_ids]].transpose(1,0,2)  # (#window, #ch, window_size+2padding)

#     ## find nan in signal segs

#     nan2d = np.any(np.isnan(signal_segs), axis=2)
#     nan1d = np.where(np.any(nan2d, axis=1))[0]
#     for i in nan1d:
#         seg_masks[i] = '%s'%(seg_mask_explanation[3])

#     return signal_segs, seg_masks, start_ids